# Editorial Review Board | Parallelization

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
class ParallelState(TypedDict):
    input: str
    tone_analysis: str
    fact_check: str
    grammar_review: str
    final_report: str

In [5]:
# Parallel workers
def analyze_tone(state: ParallelState) -> dict:
    response = model.invoke(f"Analyze the tone of this text:\n\n{state['input']}")
    return {"tone_analysis": response.content}

def check_facts(state: ParallelState) -> dict:
    response = model.invoke(f"Fact-check this text:\n\n{state['input']}")
    return {"fact_check": response.content}

def review_grammar(state: ParallelState) -> dict:
    response = model.invoke(f"Review grammar and style:\n\n{state['input']}")
    return {"grammar_review": response.content}

# Aggregator
def aggregate_results(state: ParallelState) -> dict:
    response = model.invoke(
        f"Combine these reviews into a single editorial report:\n\n"
        f"Tone: {state['tone_analysis']}\n\n"
        f"Facts: {state['fact_check']}\n\n"
        f"Grammar: {state['grammar_review']}"
    )
    return {"final_report": response.content}

In [6]:
# Build the graph with parallel fan-out
graph = StateGraph(ParallelState)
graph.add_node("tone", analyze_tone)
graph.add_node("facts", check_facts)
graph.add_node("grammar", review_grammar)
graph.add_node("aggregate", aggregate_results)

# Fan out from START to all three workers (parallel fan-out supported in LangGraph v1.x)
graph.add_edge(START, "tone")
graph.add_edge(START, "facts")
graph.add_edge(START, "grammar")

# All workers converge to aggregator (LangGraph waits for all branches before running it)
graph.add_edge("tone", "aggregate")
graph.add_edge("facts", "aggregate")
graph.add_edge("grammar", "aggregate")
graph.add_edge("aggregate", END)

parallel = graph.compile()

In [7]:
# Plot the workflow
plot_mermaid(parallel)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	tone(tone)
	facts(facts)
	grammar(grammar)
	aggregate(aggregate)
	__end__([<p>__end__</p>]):::last
	__start__ --> facts;
	__start__ --> grammar;
	__start__ --> tone;
	facts --> aggregate;
	grammar --> aggregate;
	tone --> aggregate;
	aggregate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [8]:
result = parallel.invoke({"input": "AI will replace all jobs by 2030."})
print(result["final_report"])

**Editorial Report on "AI Will Replace All Jobs by 2030"**

The statement "AI will replace all jobs by 2030" is characterized by an alarmist and speculative tone, aiming to provoke concern about the future of employment. Its definitive claim suggests a total impact, leveraging words like "all" to dramatize the issue. The urgency introduced by the timeframe "by 2030" may incite a strong emotional reaction, encouraging immediate consideration of its implications. This tone is assertive and cautionary, possibly exaggerating to elicit fear or concern.

Upon examining the facts, this claim lacks support from current data and expert consensus. While AI and automation are poised to reshape the workforce by undertaking specific tasks and revolutionizing industries, a scenario where all jobs are replaced is improbable. The complexity of jobs involving human interaction, creativity, and emotional intelligence poses significant challenges for AI replication. Historically, technological advancemen

In [9]:
stream_invoke(parallel, {"input": "AI will replace all jobs by 2030."})

────────────────────────────────────────────────────────────────────────────────

  STREAMING EXECUTION

────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────

  EXECUTION COMPLETE

────────────────────────────────────────────────────────────────────────────────

{'input': 'AI will replace all jobs by 2030.',
 'tone_analysis': 'The tone of the text "AI will replace all jobs by 2030" can be characterized as alarmist and speculative. It presents a definitive and urgent claim that may provoke concern or fear regarding the future of employment. The use of the word "all" suggests a complete and total impact, which adds to the dramatic and possibly exaggerated nature of the statement. Additionally, the specific timeframe of "by 2030" may create a sense of immediacy, urging readers to consider the implications in a relatively short period of time. Overall, the tone is assertive, cautionary, and possibly intended to provoke a strong emotional reaction.',
 'fact_check': 'The claim that AI will replace all jobs by 2030 is not supported by current data or expert consensus. While AI and automation are expected to significantly impact the workforce by taking over certain tasks and transforming industries, it is unlikely that all jobs will be replaced. Many 